# SQLite vs PostgreSQL vs MySQL

## Quick notes
### What this covers
- A side-by-side cheat sheet of the places where SQLite, PostgreSQL and MySQL differ in daily work.
- Types, quoting, strings, case rules, paging, booleans, dates, upsert, `RETURNING`, identity columns, `ALTER TABLE`.
- JSON, full-text search, window/CTE/`FILTER`/`LATERAL`/`FULL JOIN`/`ROLLUP`, `generate_series`, procedures and triggers.
- Transactions, isolation, locking, `EXPLAIN`, permissions, concurrency, and when to pick each.
- A translation table of 25+ idioms at the end.
### Key points
- Claims about PostgreSQL are for **PostgreSQL 16**. Claims about MySQL are for **MySQL 8.0 / 8.4** (InnoDB). Claims about SQLite are for **3.45** and are run live in this notebook. Newer or older versions can differ: check your version.
- There is no PostgreSQL or MySQL server here. Every runnable SQL cell is SQLite. The other dialects are shown two ways: in tables (documented behaviour) and by compiling SQLAlchemy constructs for each dialect (text only, nothing executes).
- The "core" (SELECT, JOIN, GROUP BY, subqueries, CTEs, window functions) is shared. Most pain is in types, dates, strings, upsert, DDL and locking.
- SQLite is an embedded library with one file. PostgreSQL and MySQL are client/server databases with users, roles and many concurrent writers.
### Gotchas (the part worth revising twice)
- SQLite will happily store text in an integer column. PostgreSQL will reject it. Tests that pass on SQLite can fail on PostgreSQL.
- `||` is string concatenation in SQLite and PostgreSQL but means logical OR in default MySQL. Use `CONCAT()` when you must support MySQL.
- MySQL's default text comparison is case-insensitive. PostgreSQL's and SQLite's are case-sensitive for `=`.
- `7/2` is `3` in SQLite and PostgreSQL (integer division) but `3.5000` in MySQL.
- SQLite lets you select a bare column that is not in `GROUP BY`. PostgreSQL (and default MySQL) reject it, unless the column is determined by a grouped primary key.
- SQLite: a double-quoted word that is not a column silently becomes a string. PostgreSQL raises an error.
### When it matters
- You prototype on SQLite and deploy on PostgreSQL/MySQL, or write a library/analytics SQL that must run on several engines.
- You move data or queries between engines, or you answer interview questions on "how does X differ from Y".
### One-line recap example
```sql
-- SQLite:  SELECT name || '!' FROM t LIMIT 5;      -- PostgreSQL: same
-- MySQL:   SELECT CONCAT(name, '!') FROM t LIMIT 5;
```

## 01. How to read this notebook
- Each section has a short table of differences, a runnable SQLite cell, and where it helps, a SQLAlchemy compile showing the text each dialect would emit.
- The compile cells never connect to anything. They only turn a Python expression into SQL text.
- Versions used for claims: SQLite 3.45, PostgreSQL 16, MySQL 8.0 / 8.4. See `02-Writing-Portable-SQL` for how to write queries that work on all three.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
# helper used by the compile cells below
import sqlite3
import sqlalchemy as sa
from sqlalchemy.dialects import postgresql as pg, mysql as my, sqlite as sl
from sqlalchemy.schema import CreateTable, CreateIndex

DIALECTS = {"sqlite": sl.dialect(), "postgresql": pg.dialect(), "mysql": my.dialect()}

def show(stmt, literal=True):
    """Print the SQL a statement/DDL compiles to for each dialect (one line each)."""
    kw = {"compile_kwargs": {"literal_binds": True}} if literal else {}
    for name, d in DIALECTS.items():
        try:
            text = " ".join(str(stmt.compile(dialect=d, **kw)).split())
        except Exception as e:
            text = f"<cannot compile: {type(e).__name__}>"
        print(f"{name:11}| {text}")

print("SQLAlchemy", sa.__version__, "| SQLite library", sqlite3.sqlite_version)

SQLAlchemy 2.0.34 | SQLite library 3.45.3


## 02. Type systems
- **SQLite**: flexible ("dynamic") typing. A column has a type *affinity* (INTEGER, TEXT, REAL, NUMERIC, BLOB), not a hard type. Any value can go in any column unless the table is `STRICT` (3.37+).
- **PostgreSQL**: strict typing. Rich types: `integer`, `bigint`, `numeric(p,s)`, `text`, `boolean`, `date`, `timestamp`/`timestamptz`, `uuid`, `json`/`jsonb`, arrays (`int[]`), ranges, enums.
- **MySQL**: strict types in default strict mode (`STRICT_TRANS_TABLES`). `INT`/`TINYINT`/`BIGINT`, `DECIMAL(p,s)`, `VARCHAR(n)`, `TEXT`, `DATETIME`/`TIMESTAMP`, `JSON`, `ENUM`. No real boolean (`BOOL` = `TINYINT(1)`), no arrays, no native uuid type (use `CHAR(36)` or `BINARY(16)`).
- Integer division: `7/2` is `3` in SQLite and PostgreSQL; MySQL's `/` always gives a decimal (`3.5000`) and `DIV` is integer division.
- Division by zero: SQLite returns NULL; PostgreSQL raises an error; MySQL returns NULL in a plain SELECT (it can error in strict-mode writes).

In [3]:
%%sql
CREATE TABLE loose (a INTEGER, b TEXT, c REAL);
INSERT INTO loose VALUES ('12', 'x', 3), ('abc', 5, '2.5');
SELECT a, typeof(a) AS ta, b, typeof(b) AS tb, c, typeof(c) AS tc FROM loose ORDER BY rowid;

a   | ta      | b | tb   | c   | tc  
----+---------+---+------+-----+-----
12  | integer | x | text | 3.0 | real
abc | text    | 5 | text | 2.5 | real
(2 rows)



- `'12'` became the integer 12 (the column has INTEGER affinity and the text looks like a number). `'abc'` stayed text. `5` in the TEXT column became the text `'5'`. No error anywhere.
- A `STRICT` table (3.37+) behaves much more like PostgreSQL:

In [4]:
%%sql expect_error
CREATE TABLE strict_t (a INTEGER) STRICT;
INSERT INTO strict_t VALUES ('abc');

ERROR: cannot store TEXT value in INTEGER column strict_t.a



In [5]:
%%sql
SELECT 7 / 2 AS int_div, 7.0 / 2 AS real_div, 7 % 2 AS modulo, 1 / 0 AS div_zero;

int_div | real_div | modulo | div_zero
--------+----------+--------+---------
3       | 3.5      | 1      | NULL    
(1 row)



- Next, the same logical table compiled by SQLAlchemy for each dialect. Watch the column types change (`BOOLEAN` vs `BOOL`, `UUID` vs `CHAR(32)`, `TIMESTAMP WITHOUT TIME ZONE` vs `DATETIME`, `SERIAL` vs `AUTO_INCREMENT`).

In [6]:
meta = sa.MetaData()
demo = sa.Table("demo", meta,
                sa.Column("id", sa.Integer, primary_key=True),
                sa.Column("price", sa.Numeric(10, 2)),
                sa.Column("is_ok", sa.Boolean),
                sa.Column("seen", sa.DateTime),
                sa.Column("ref", sa.Uuid),
                sa.Column("doc", sa.JSON))
for name, d in DIALECTS.items():
    print(f"--- {name}")
    print(str(CreateTable(demo).compile(dialect=d)).strip())

--- sqlite
CREATE TABLE demo (
	id INTEGER NOT NULL, 
	price NUMERIC(10, 2), 
	is_ok BOOLEAN, 
	seen DATETIME, 
	ref CHAR(32), 
	doc JSON, 
	PRIMARY KEY (id)
)
--- postgresql
CREATE TABLE demo (
	id SERIAL NOT NULL, 
	price NUMERIC(10, 2), 
	is_ok BOOLEAN, 
	seen TIMESTAMP WITHOUT TIME ZONE, 
	ref UUID, 
	doc JSON, 
	PRIMARY KEY (id)
)
--- mysql
CREATE TABLE demo (
	id INTEGER NOT NULL AUTO_INCREMENT, 
	price NUMERIC(10, 2), 
	is_ok BOOL, 
	seen DATETIME, 
	ref CHAR(32), 
	doc JSON, 
	PRIMARY KEY (id)
)


## 03. Identifiers and quoting
- Standard SQL: `"double quotes"` for identifiers, `'single quotes'` for strings. SQLite and PostgreSQL follow this.
- MySQL: backticks `` `name` `` for identifiers. Double quotes mean a *string* unless the `ANSI_QUOTES` SQL mode is on. (SQLite also accepts backticks and `[brackets]` for MySQL/SQL Server compatibility.)
- Unquoted names: PostgreSQL folds them to lower case (`Users` becomes `users`; quote it to keep the capitals). SQLite ignores case in names. MySQL table-name case depends on the operating system and the `lower_case_table_names` setting.
- Quote an identifier only when it is a reserved word or has odd characters. Prefer lower_snake_case names that need no quotes anywhere.
- SQLite misfeature: if `"nonexistent"` is not a column, SQLite treats it as the string `'nonexistent'`. A typo in a column name can turn into a silent constant.

In [7]:
%%sql
CREATE TABLE "order" ("select" INTEGER, "Total Amount" REAL);
INSERT INTO "order" VALUES (1, 9.5);
SELECT "select", "Total Amount", `Total Amount` AS backtick_works, "nonexistent" AS typo FROM "order";

select | Total Amount | backtick_works | typo       
-------+--------------+----------------+------------
1      | 9.5          | 9.5            | nonexistent
(1 row)



In [8]:
from sqlalchemy.schema import CreateTable
reserved = sa.Table("order", sa.MetaData(), sa.Column("select", sa.Integer), sa.Column("Total Amount", sa.Float))
show(sa.select(reserved.c["select"], reserved.c["Total Amount"]))

sqlite     | SELECT "order"."select", "order"."Total Amount" FROM "order"
postgresql | SELECT "order"."select", "order"."Total Amount" FROM "order"
mysql      | SELECT `order`.`select`, `order`.`Total Amount` FROM `order`


## 04. Strings: concatenation, length, search
| Task | SQLite | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|
| Concatenate | `a \|\| b`, `concat()` (3.44+) | `a \|\| b`, `concat()` | `CONCAT(a, b)` (`\|\|` is OR by default) |
| NULL in concat | `\|\|` gives NULL; `concat()` skips NULL | `\|\|` gives NULL; `concat()` skips NULL | `CONCAT` gives NULL; `CONCAT_WS` skips NULL |
| Length in characters | `length(s)` | `length(s)` / `char_length(s)` | `CHAR_LENGTH(s)` (`LENGTH` = bytes) |
| Position | `instr(s, sub)` | `position(sub in s)`, `strpos(s, sub)` | `INSTR(s, sub)`, `LOCATE(sub, s)` |
| Substring | `substr(s, start, n)` | `substr` / `substring(s from 1 for n)` | `SUBSTRING` / `SUBSTR` |
| Join rows into one string | `group_concat(x, ',')` | `string_agg(x, ',')` | `GROUP_CONCAT(x SEPARATOR ',')` |
- Portable choice for concatenation across all three: `CONCAT()` works in PostgreSQL and MySQL, and SQLite 3.44+, and it skips NULLs on SQLite/PostgreSQL but not MySQL. If NULL behaviour matters, wrap each part in `COALESCE(x, '')`.

In [9]:
%%sql
SELECT 'a' || NULL AS pipe_null,
       concat('a', NULL, 'b') AS concat_skips_null,
       concat_ws('-', 'a', NULL, 'b') AS ws_skips_null,
       length('héllo') AS chars,
       length(CAST('héllo' AS BLOB)) AS bytes,
       instr('hello', 'l') AS first_l,
       substr('hello', 2, 3) AS sub;
CREATE TABLE tag (post INTEGER, name TEXT);
INSERT INTO tag VALUES (1,'sql'),(1,'db'),(2,'py');
SELECT post, group_concat(name, ',') AS tags FROM (SELECT * FROM tag ORDER BY post, name) GROUP BY post ORDER BY post;

pipe_null | concat_skips_null | ws_skips_null | chars | bytes | first_l | sub
----------+-------------------+---------------+-------+-------+---------+----
NULL      | ab                | a-b           | 5     | 6     | 3       | ell
(1 row)

post | tags  
-----+-------
1    | db,sql
2    | py    
(2 rows)



In [10]:
t = sa.table("t", sa.column("name", sa.String), sa.column("surname", sa.String))
show(sa.select(t.c.name + " " + t.c.surname))
show(sa.select(sa.func.concat(t.c.name, t.c.surname)))

sqlite     | SELECT t.name || ' ' || t.surname AS anon_1 FROM t
postgresql | SELECT t.name || ' ' || t.surname AS anon_1 FROM t
mysql      | SELECT concat(t.name, ' ', t.surname) AS anon_1 FROM t
sqlite     | SELECT concat(t.name, t.surname) AS concat_1 FROM t
postgresql | SELECT concat(t.name, t.surname) AS concat_1 FROM t
mysql      | SELECT concat(t.name, t.surname) AS concat_1 FROM t


- In SQLAlchemy the `+` operator on strings becomes `||` on SQLite and PostgreSQL and `concat(...)` on MySQL. That is the kind of difference an abstraction layer can hide.
- The order inside `group_concat`/`string_agg` is not guaranteed unless you specify it (PostgreSQL: `string_agg(x, ',' ORDER BY x)`; MySQL: `GROUP_CONCAT(x ORDER BY x SEPARATOR ',')`). SQLite: order a subquery first, as above, which usually works but is not a documented guarantee.

## 05. Case sensitivity and collations
- **SQLite**: `=` is case-sensitive by default (the `BINARY` collation). `LIKE` is case-insensitive for ASCII letters only. `COLLATE NOCASE` makes `=` case-insensitive (ASCII only).
- **PostgreSQL**: `=` is case-sensitive. `LIKE` is case-sensitive; `ILIKE` is the case-insensitive version. For case-insensitive columns use the `citext` extension, `lower()` comparisons, or a nondeterministic collation (PostgreSQL 12+; check which operators it supports in your version).
- **MySQL**: the default collation in 8.0 is `utf8mb4_0900_ai_ci`, which is case- and accent-insensitive, so `'abc' = 'ABC'` is true. Pick a `_bin` or `_as_cs` collation for case-sensitive comparison.
- The same query can return a different number of rows on different engines just because of this.

In [11]:
%%sql
SELECT 'abc' = 'ABC' AS eq,
       'abc' LIKE 'ABC' AS like_ascii,
       'é' LIKE 'É' AS like_non_ascii,
       'abc' = 'ABC' COLLATE NOCASE AS eq_nocase;
CREATE TABLE person (name TEXT COLLATE NOCASE);
INSERT INTO person VALUES ('Ann'), ('ann'), ('BOB');
SELECT name FROM person WHERE name = 'ANN' ORDER BY rowid;

eq | like_ascii | like_non_ascii | eq_nocase
---+------------+----------------+----------
0  | 1          | 0              | 1        
(1 row)

name
----
Ann 
ann 
(2 rows)



In [12]:
col = sa.column("name", sa.String)
show(sa.select(col).where(col.ilike("a%")))
show(sa.select(col).where(col.contains("x")))

sqlite     | SELECT name WHERE lower(name) LIKE lower('a%')
postgresql | SELECT name WHERE name ILIKE 'a%%'
mysql      | SELECT name FROM DUAL WHERE lower(name) LIKE lower('a%%')
sqlite     | SELECT name WHERE (name LIKE '%' || 'x' || '%')
postgresql | SELECT name WHERE (name LIKE '%%' || 'x' || '%%')
mysql      | SELECT name FROM DUAL WHERE (name LIKE concat('%%', 'x', '%%'))


- `ilike` compiles to a real `ILIKE` on PostgreSQL but to `lower(...) LIKE lower(...)` on SQLite and MySQL. (The doubled `%%` in the PostgreSQL and MySQL lines is only the Python driver's escaping of a literal percent sign.) The `lower()` form is the portable idiom, at the cost of not using a plain index (see `07-Performance`).

## 06. Booleans
- **PostgreSQL** has a real `boolean` type (`true`/`false`/NULL). You cannot `SUM()` a boolean; use `count(*) FILTER (WHERE flag)` or cast.
- **MySQL**: `BOOLEAN` is an alias for `TINYINT(1)`. `TRUE` and `FALSE` are just 1 and 0.
- **SQLite**: no boolean type. `TRUE`/`FALSE` keywords (3.23+) are the integers 1 and 0.
- Portable habit: store flags as a boolean where it exists, and test them with `WHERE flag` or `flag = TRUE`, not `flag = 1`. PostgreSQL rejects `boolean = integer`.

In [13]:
%%sql
SELECT TRUE AS t, FALSE AS f, typeof(TRUE) AS type_of_true, 1 = TRUE AS one_equals_true;
CREATE TABLE feature (name TEXT, enabled BOOLEAN CHECK (enabled IN (0, 1)));
INSERT INTO feature VALUES ('dark', TRUE), ('beta', FALSE), ('ai', TRUE);
SELECT name FROM feature WHERE enabled ORDER BY name;
SELECT SUM(enabled) AS on_count FROM feature;

t | f | type_of_true | one_equals_true
--+---+--------------+----------------
1 | 0 | integer      | 1              
(1 row)

name
----
ai  
dark
(2 rows)

on_count
--------
2       
(1 row)



## 07. Limiting rows: LIMIT, OFFSET, FETCH FIRST
| | SQLite | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|
| First n rows | `LIMIT n` | `LIMIT n` or `FETCH FIRST n ROWS ONLY` | `LIMIT n` |
| Skip rows | `LIMIT n OFFSET m` | `OFFSET m ROWS FETCH FIRST n ROWS ONLY` | `LIMIT n OFFSET m` or `LIMIT m, n` |
| Ties | no | `FETCH FIRST n ROWS WITH TIES` | no |
- `FETCH FIRST` is the SQL-standard form. SQLite and MySQL do not support it. `LIMIT/OFFSET` works on all three.
- Note the MySQL `LIMIT m, n` form is *offset first*, the reverse of reading order. SQLite also accepts it. Avoid it.
- Deep `OFFSET` pages get slower on every engine because skipped rows are still read. Prefer keyset pagination (see `04-Patterns`).

In [14]:
%%sql
CREATE TABLE post (id INTEGER PRIMARY KEY, title TEXT);
INSERT INTO post VALUES (1,'a'),(2,'b'),(3,'c'),(4,'d'),(5,'e');
SELECT id FROM post ORDER BY id LIMIT 2 OFFSET 2;
SELECT id FROM post ORDER BY id LIMIT 2, 2;

id
--
3 
4 
(2 rows)

id
--
3 
4 
(2 rows)



In [15]:
p = sa.table("post", sa.column("id", sa.Integer))
show(sa.select(p.c.id).order_by(p.c.id).limit(2).offset(2))
show(sa.select(p.c.id).order_by(p.c.id).fetch(2, with_ties=True))

sqlite     | SELECT post.id FROM post ORDER BY post.id LIMIT 2 OFFSET 2
postgresql | SELECT post.id FROM post ORDER BY post.id LIMIT 2 OFFSET 2
mysql      | SELECT post.id FROM post ORDER BY post.id LIMIT 2, 2
sqlite     | SELECT post.id FROM post ORDER BY post.id FETCH FIRST 2 ROWS WITH TIES
postgresql | SELECT post.id FROM post ORDER BY post.id FETCH FIRST (2) ROWS WITH TIES
mysql      | SELECT post.id FROM post ORDER BY post.id FETCH FIRST 2 ROWS WITH TIES


- Look at the compile: SQLite and PostgreSQL emit `LIMIT 2 OFFSET 2`, MySQL emits `LIMIT 2, 2` (offset first). SQLAlchemy will also generate `FETCH FIRST ... ROWS WITH TIES` text, but only PostgreSQL (13+) actually runs it; do not rely on it for MySQL or SQLite.

## 08. NULL ordering
- SQLite and MySQL treat NULL as *smaller* than any value: `ORDER BY x ASC` puts NULLs first.
- PostgreSQL treats NULL as *larger*: `ORDER BY x ASC` puts NULLs last.
- `NULLS FIRST` / `NULLS LAST` exist in PostgreSQL and SQLite 3.30+, not in MySQL. In MySQL emulate with `ORDER BY x IS NULL, x`.
- Always say what you want when NULLs are possible, so all engines agree.

In [16]:
%%sql
CREATE TABLE score (who TEXT, pts INTEGER);
INSERT INTO score VALUES ('a', 10), ('b', NULL), ('c', 5);
SELECT who, pts FROM score ORDER BY pts;
SELECT who, pts FROM score ORDER BY pts NULLS LAST;
SELECT who, pts FROM score ORDER BY pts IS NULL, pts;

who | pts 
----+-----
b   | NULL
c   | 5   
a   | 10  
(3 rows)

who | pts 
----+-----
c   | 5   
a   | 10  
b   | NULL
(3 rows)

who | pts 
----+-----
c   | 5   
a   | 10  
b   | NULL
(3 rows)



## 09. Date and time functions
| Task | SQLite | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|
| Current timestamp | `CURRENT_TIMESTAMP` (UTC text) | `now()`, `CURRENT_TIMESTAMP` | `NOW()`, `CURRENT_TIMESTAMP` |
| Format | `strftime('%Y-%m', d)` | `to_char(d, 'YYYY-MM')` | `DATE_FORMAT(d, '%Y-%m')` |
| Truncate to month | `date(d, 'start of month')` | `date_trunc('month', d)` | `DATE_FORMAT(d, '%Y-%m-01')` |
| Add a day | `date(d, '+1 day')` | `d + interval '1 day'` | `DATE_ADD(d, INTERVAL 1 DAY)` |
| Difference in days | `julianday(a) - julianday(b)` | `a - b` (for dates), `age()` | `DATEDIFF(a, b)` |
| Extract part | `strftime('%Y', d)` (text) | `extract(year from d)` | `EXTRACT(YEAR FROM d)`, `YEAR(d)` |
- SQLite has no date type. Dates are text (`'2024-03-15'`), numbers (Julian day, Unix time) and the functions interpret them. Store ISO-8601 text so sorting works.
- Month arithmetic differs at month ends: `2024-01-31 + 1 month`. In SQLite the result overflows to `2024-03-02`. PostgreSQL clamps to the last day of the month (`2024-02-29`), and MySQL's `DATE_ADD` clamps too. Test month-end cases on your engine.

In [17]:
%%sql
SELECT strftime('%Y-%m', '2024-03-15') AS ym,
       date('2024-03-15', 'start of month') AS month_start,
       date('2024-01-31', '+1 month') AS month_end_overflow,
       julianday('2024-03-01') - julianday('2024-02-01') AS days_between,
       strftime('%w', '2024-03-15') AS weekday_0_sunday,
       strftime('%Y-%m-%d %H:%M', '2024-03-15 13:45:10') AS short_ts;

ym      | month_start | month_end_overflow | days_between | weekday_0_sunday | short_ts        
--------+-------------+--------------------+--------------+------------------+-----------------
2024-03 | 2024-03-01  | 2024-03-02         | 29.0         | 5                | 2024-03-15 13:45
(1 row)



In [18]:
d = sa.column("d", sa.Date)
show(sa.select(sa.func.date_trunc("month", d)))
show(sa.select(sa.func.now()))
show(sa.select(sa.extract("year", d)))

sqlite     | SELECT date_trunc('month', d) AS date_trunc_1
postgresql | SELECT date_trunc('month', d) AS date_trunc_1
mysql      | SELECT date_trunc('month', d) AS date_trunc_1
sqlite     | SELECT CURRENT_TIMESTAMP AS now_1
postgresql | SELECT now() AS now_1
mysql      | SELECT now() AS now_1
sqlite     | SELECT CAST(STRFTIME('%Y', d) AS INTEGER) AS anon_1
postgresql | SELECT EXTRACT(year FROM d) AS anon_1
mysql      | SELECT EXTRACT(year FROM d) AS anon_1


- `sa.extract("year", d)` is portable text on all three (SQLAlchemy rewrites it for SQLite as `CAST(STRFTIME('%Y', d) AS INTEGER)`). `date_trunc` is a PostgreSQL function; SQLAlchemy just passes the name through, so it compiles on every dialect but only *runs* on PostgreSQL. Compiling is not the same as being valid.

## 10. Upsert (insert or update)
| | Syntax |
|---|---|
| SQLite (3.24+) | `INSERT ... ON CONFLICT (key) DO UPDATE SET c = excluded.c` |
| PostgreSQL | same: `ON CONFLICT (key) DO UPDATE SET c = EXCLUDED.c` (and `DO NOTHING`). PG 15+ also has `MERGE` |
| MySQL | `INSERT ... ON DUPLICATE KEY UPDATE c = VALUES(c)` (row-alias style, 8.0.19+: `... AS new ON DUPLICATE KEY UPDATE c = new.c`; `VALUES(c)` is deprecated since 8.0.20 but still works), plus `REPLACE INTO` and `INSERT IGNORE` |
- `ON CONFLICT` needs you to name the unique key (or constraint). MySQL fires on *any* unique key violation.
- `REPLACE INTO` in SQLite and MySQL deletes the old row and inserts a new one. That resets columns you did not mention. In MySQL it fires delete triggers; in SQLite only if `recursive_triggers` is on. Prefer a real upsert. See `06-Data-Modification` for the SQLite details.

In [19]:
%%sql
CREATE TABLE stock (sku TEXT PRIMARY KEY, qty INTEGER);
INSERT INTO stock VALUES ('A', 5);
INSERT INTO stock VALUES ('A', 3), ('B', 7)
  ON CONFLICT (sku) DO UPDATE SET qty = stock.qty + excluded.qty;
SELECT * FROM stock ORDER BY sku;

sku | qty
----+----
A   | 8  
B   | 7  
(2 rows)



In [20]:
from sqlalchemy.dialects.postgresql import insert as pg_insert
from sqlalchemy.dialects.mysql import insert as my_insert
from sqlalchemy.dialects.sqlite import insert as sl_insert
stock = sa.Table("stock", sa.MetaData(), sa.Column("sku", sa.String, primary_key=True), sa.Column("qty", sa.Integer))
i = pg_insert(stock).values(sku="A", qty=3)
print("postgresql|", " ".join(str(i.on_conflict_do_update(index_elements=["sku"], set_={"qty": stock.c.qty + i.excluded.qty}).compile(dialect=pg.dialect(), compile_kwargs={"literal_binds": True})).split()))
i = sl_insert(stock).values(sku="A", qty=3)
print("sqlite    |", " ".join(str(i.on_conflict_do_update(index_elements=["sku"], set_={"qty": stock.c.qty + i.excluded.qty}).compile(dialect=sl.dialect(), compile_kwargs={"literal_binds": True})).split()))
i = my_insert(stock).values(sku="A", qty=3)
print("mysql     |", " ".join(str(i.on_duplicate_key_update(qty=stock.c.qty + i.inserted.qty).compile(dialect=my.dialect(), compile_kwargs={"literal_binds": True})).split()))

postgresql| INSERT INTO stock (sku, qty) VALUES ('A', 3) ON CONFLICT (sku) DO UPDATE SET qty = (stock.qty + excluded.qty)
sqlite    | INSERT INTO stock (sku, qty) VALUES ('A', 3) ON CONFLICT (sku) DO UPDATE SET qty = (stock.qty + excluded.qty)
mysql     | INSERT INTO stock (sku, qty) VALUES ('A', 3) ON DUPLICATE KEY UPDATE qty = (stock.qty + VALUES(qty))


- Same intent, two spellings (MySQL's `VALUES(qty)` is the older form, deprecated since 8.0.20). Full recipes, including the `DO NOTHING` form, are in `02-Writing-Portable-SQL`.

## 11. RETURNING and auto-increment / identity
- **RETURNING**: PostgreSQL yes (all of INSERT/UPDATE/DELETE). SQLite 3.35+ yes. MySQL 8.0/8.4 no (MariaDB 10.5+ has it). On MySQL you read `LAST_INSERT_ID()` or the driver's `lastrowid` after the insert.
- **Auto-increment keys**:
  - SQLite: `INTEGER PRIMARY KEY` is an alias for the internal rowid and auto-assigns. `AUTOINCREMENT` only adds "never reuse an id".
  - PostgreSQL: `GENERATED ... AS IDENTITY` (standard, v10+, preferred) or the older `SERIAL`/`BIGSERIAL`. Both use a sequence, so ids can have gaps (rolled-back inserts still burn numbers).
  - MySQL: `AUTO_INCREMENT` on a key column.
- Never assume ids are gapless on any engine.

In [21]:
%%sql
CREATE TABLE note (id INTEGER PRIMARY KEY, body TEXT);
INSERT INTO note (body) VALUES ('first') RETURNING id, body;
UPDATE note SET body = upper(body) WHERE id = 1 RETURNING id, body;

id | body 
---+------
1  | first
(1 row)

id | body 
---+------
1  | FIRST
(1 row)



In [22]:
note = sa.Table("note", sa.MetaData(), sa.Column("id", sa.Integer, primary_key=True), sa.Column("body", sa.String))
show(sa.insert(note).values(body="x").returning(note.c.id))
print()
big = sa.Table("big", sa.MetaData(), sa.Column("id", sa.BigInteger, sa.Identity(), primary_key=True))
for name, d in DIALECTS.items():
    print(f"{name:11}|", " ".join(str(CreateTable(big).compile(dialect=d)).split()))

sqlite     | INSERT INTO note (body) VALUES ('x') RETURNING id
postgresql | INSERT INTO note (body) VALUES ('x') RETURNING note.id
mysql      | INSERT INTO note (body) VALUES ('x') RETURNING note.id

sqlite     | CREATE TABLE big ( id BIGINT NOT NULL, PRIMARY KEY (id) )
postgresql | CREATE TABLE big ( id BIGINT GENERATED BY DEFAULT AS IDENTITY, PRIMARY KEY (id) )
mysql      | CREATE TABLE big ( id BIGINT NOT NULL AUTO_INCREMENT, PRIMARY KEY (id) )


- Careful with the first compile line: SQLAlchemy prints a `RETURNING` clause even for the MySQL dialect object. That is SQLAlchemy's generic text, not proof MySQL accepts it. MySQL 8 does not support `RETURNING`.
- In the DDL compile, PostgreSQL gets `GENERATED BY DEFAULT AS IDENTITY`; MySQL drops the `Identity()` and uses `AUTO_INCREMENT`. SQLite also drops it and emits `BIGINT NOT NULL, PRIMARY KEY (id)`. In SQLite only a column declared exactly `INTEGER PRIMARY KEY` auto-assigns ids, so this `BIGINT` key would not (see `02-Writing-Portable-SQL`, section 06).

## 12. ALTER TABLE and DDL transactions
| | SQLite 3.45 | PostgreSQL 16 | MySQL 8 (InnoDB) |
|---|---|---|---|
| `ADD COLUMN` | yes (limits on defaults/constraints) | yes | yes |
| `RENAME COLUMN` / `RENAME TO` | yes (3.25+) | yes | yes |
| `DROP COLUMN` | yes (3.35+, with restrictions) | yes | yes |
| Change column type / `ALTER COLUMN` | **no** (rebuild the table) | yes (`ALTER COLUMN ... TYPE`) | yes (`MODIFY` / `CHANGE`) |
| Add/drop constraints later | mostly no (rebuild) | yes | yes |
| DDL inside a transaction | yes, rolls back | yes, rolls back | no: most DDL commits implicitly |
- Transactional DDL is a big PostgreSQL/SQLite strength: a failed migration leaves nothing half-done. On MySQL 8 each single DDL statement is atomic, but it commits implicitly, so a failed multi-step migration can leave you half-migrated. (PostgreSQL has exceptions too, for example `CREATE INDEX CONCURRENTLY` cannot run in a transaction.)
- The SQLite "rebuild" recipe (create new, copy, drop, rename) is in `05-DDL-And-Integrity` and `09-Design`.

In [23]:
%%sql
CREATE TABLE acct (id INTEGER PRIMARY KEY, name TEXT);
BEGIN;
ALTER TABLE acct ADD COLUMN email TEXT;
ALTER TABLE acct RENAME COLUMN name TO full_name;
ROLLBACK;
SELECT name FROM pragma_table_info('acct') ORDER BY cid;

name
----
id  
name
(2 rows)



In [24]:
%%sql expect_error
ALTER TABLE acct ALTER COLUMN full_name TYPE VARCHAR(100);

ERROR: near "ALTER": syntax error



- The `ROLLBACK` undid both ALTERs (the columns are still `id`, `name`), which shows SQLite DDL is transactional. The last cell shows the PostgreSQL-style `ALTER COLUMN ... TYPE` is a syntax error in SQLite.

## 13. JSON
| | SQLite 3.45 | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|
| Type | text with JSON functions (3.45+ can also keep a binary `JSONB` blob, but there is no JSON column type) | `json` and `jsonb` (indexable binary) | `JSON` (validated, binary) |
| Get as JSON | `->` (3.38+) | `->` | `->` (shorthand for `JSON_EXTRACT`) |
| Get as text/scalar | `->>` | `->>` | `->>` (extract + unquote) |
| Path style | `'$.a.b'` | `'{a,b}'` with `#>`, or chain `->` | `'$.a.b'` |
| Contains | none built in (use `json_each`) | `@>`, `?` on `jsonb`, GIN index | `JSON_CONTAINS` |
| Rows from array | `json_each()` | `jsonb_array_elements()` | `JSON_TABLE()` |
- `->` and `->>` exist in all three, but the right-hand side differs: SQLite and MySQL expect a JSON path (`'$.a'`; SQLite also accepts a bare key); PostgreSQL takes a key or index (`'a'`).
- See `08-SQLite-Specials` for the full SQLite JSON tour.

In [25]:
%%sql
CREATE TABLE ev (id INTEGER PRIMARY KEY, doc TEXT);
INSERT INTO ev VALUES (1, '{"user":{"name":"Ann","age":31},"tags":["a","b"]}'),
                      (2, '{"user":{"name":"Bob"},"tags":[]}');
SELECT id, doc -> '$.user' AS user_json, doc ->> '$.user.name' AS name, doc ->> '$.user.age' AS age FROM ev ORDER BY id;
SELECT ev.id, j.value AS tag FROM ev, json_each(ev.doc, '$.tags') AS j ORDER BY ev.id, j.key;

id | user_json               | name | age 
---+-------------------------+------+-----
1  | {"name":"Ann","age":31} | Ann  | 31  
2  | {"name":"Bob"}          | Bob  | NULL
(2 rows)

id | tag
---+----
1  | a  
1  | b  
(2 rows)



In [26]:
events = sa.table("ev", sa.column("doc", sa.JSON))
show(sa.select(events.c.doc["user"]["name"].as_string()), literal=False)

sqlite     | SELECT JSON_EXTRACT(JSON_QUOTE(JSON_EXTRACT(ev.doc, ?)), ?) AS anon_1 FROM ev
postgresql | SELECT CAST((ev.doc -> %(doc_1)s) ->> %(param_1)s AS VARCHAR) AS anon_1 FROM ev
mysql      | SELECT CASE JSON_EXTRACT(JSON_EXTRACT(ev.doc, %s), %s) WHEN 'null' THEN NULL ELSE JSON_UNQUOTE(JSON_EXTRACT(JSON_EXTRACT(ev.doc, %s), %s)) END AS anon_1 FROM ev


- One SQLAlchemy expression, three different JSON syntaxes. PostgreSQL gets `->>`, MySQL needs `JSON_UNQUOTE(JSON_EXTRACT(...))` with a NULL guard, SQLite uses nested `JSON_EXTRACT`/`JSON_QUOTE` calls with paths. That is a good argument for letting a tool write JSON access.

## 14. Full-text search
- **SQLite**: the FTS5 virtual table (`CREATE VIRTUAL TABLE ... USING fts5(...)`, `WHERE docs MATCH 'word'`, `bm25()` ranking). Compiled in most builds, but check yours.
- **PostgreSQL**: `tsvector` / `tsquery` with `to_tsvector('english', col) @@ to_tsquery('english', 'word')`, indexed with GIN. The `pg_trgm` extension gives fuzzy/`LIKE '%x%'` index support.
- **MySQL**: `FULLTEXT` index and `MATCH (col) AGAINST ('word' IN NATURAL LANGUAGE MODE)`.
- None of these are portable: pick the engine's own tool. For a cross-engine app, hide it behind one function in your code. See `08-SQLite-Specials/` for FTS5 in depth.

In [27]:
%%sql
CREATE VIRTUAL TABLE docs USING fts5(title, body);
INSERT INTO docs VALUES ('SQLite notes', 'embedded database in one file'),
                        ('Postgres notes', 'server database with roles'),
                        ('Cooking', 'pasta and sauce');
SELECT title FROM docs WHERE docs MATCH 'database' ORDER BY title;

title         
--------------
Postgres notes
SQLite notes  
(2 rows)



## 15. Feature support at a glance
| Feature | SQLite 3.45 | PostgreSQL 16 | MySQL 8.0 / 8.4 |
|---|---|---|---|
| CTEs (`WITH`, `WITH RECURSIVE`) | yes | yes | yes (8.0+; not in 5.7) |
| Window functions | yes (3.25+) | yes | yes (8.0+) |
| `FILTER (WHERE ...)` on aggregates | yes (3.30+) | yes | **no**: use `SUM(CASE WHEN ...)` |
| `FULL OUTER JOIN` | yes (3.39+) | yes | **no**: `LEFT JOIN UNION ALL` anti-join |
| `LATERAL` join | **no** | yes | yes (8.0.14+) |
| `INTERSECT` / `EXCEPT` | yes | yes | yes (8.0.31+) |
| `GROUPING SETS` / `CUBE` / `ROLLUP` | **no** | all three | `GROUP BY ... WITH ROLLUP` only |
| `PIVOT` keyword | no | no (`crosstab()` in `tablefunc` extension) | no |
| `generate_series()` | no (recursive CTE; CLI extension only) | yes | no (recursive CTE) |
| `DISTINCT ON` | no | yes | no |
| `IS DISTINCT FROM` | yes (3.39+) | yes | `<=>` operator instead |
| `UPDATE ... FROM` | yes (3.33+) | yes | multi-table `UPDATE ... JOIN` instead |
| Partial indexes (`WHERE` on an index) | yes | yes | **no** |
| Expression indexes | yes | yes | yes (8.0.13+, functional key parts) |
| Arrays, ranges, custom types | no | yes | no |
- There is no `PIVOT` in any of the three. The portable pivot is conditional aggregation (see `02-Querying`, `04-Patterns`).
- `LATERAL` is "a subquery that may refer to columns of earlier tables in `FROM`". On SQLite use a correlated subquery or a window function (top-N per group).

## 16. SQLite workarounds for what it lacks
- Sales by region and month, to show conditional aggregation (pivot; `FILTER` works on SQLite and PostgreSQL, the `CASE` form works on all three), a `ROLLUP`-style subtotal, and a `generate_series` replacement.

In [28]:
%%sql
CREATE TABLE sale (region TEXT, month TEXT, amount INTEGER);
INSERT INTO sale VALUES ('N','2024-01',100),('N','2024-02',150),('S','2024-01',80),('S','2024-02',120),('S','2024-02',30);
SELECT region,
       SUM(amount) FILTER (WHERE month = '2024-01') AS jan_filter,
       SUM(CASE WHEN month = '2024-01' THEN amount END) AS jan_case,
       SUM(CASE WHEN month = '2024-02' THEN amount END) AS feb
FROM sale GROUP BY region ORDER BY region;

region | jan_filter | jan_case | feb
-------+------------+----------+----
N      | 100        | 100      | 150
S      | 80         | 80       | 150
(2 rows)



In [29]:
%%sql
-- ROLLUP emulation: detail rows + subtotal per region + grand total, with UNION ALL
SELECT region, month, SUM(amount) AS total FROM sale GROUP BY region, month
UNION ALL SELECT region, 'ALL', SUM(amount) FROM sale GROUP BY region
UNION ALL SELECT 'ALL', 'ALL', SUM(amount) FROM sale
ORDER BY region, month;

region | month   | total
-------+---------+------
ALL    | ALL     | 480  
N      | 2024-01 | 100  
N      | 2024-02 | 150  
N      | ALL     | 250  
S      | 2024-01 | 80   
S      | 2024-02 | 150  
S      | ALL     | 230  
(7 rows)



In [30]:
%%sql
-- generate_series(1, 5) replacement: recursive CTE
WITH RECURSIVE n(i) AS (SELECT 1 UNION ALL SELECT i + 1 FROM n WHERE i < 5)
SELECT i FROM n ORDER BY i;

i
-
1
2
3
4
5
(5 rows)



- On PostgreSQL: `SELECT * FROM generate_series(1, 5)` (also works with dates and an interval step). On MySQL the recursive CTE above works as is.
- On PostgreSQL the ROLLUP query is `GROUP BY ROLLUP (region, month)`; on MySQL `GROUP BY region, month WITH ROLLUP`. Both emit `NULL` (not `'ALL'`) in the subtotal columns.
- Top-N per group (the common reason for `LATERAL`) with a window function, which works on all three (note the alias on the subquery: MySQL and PostgreSQL before 16 require one; see `03-Window-Functions/01-Window-Functions`):

In [31]:
%%sql
SELECT region, month, amount FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC, month) AS rn FROM sale
) AS ranked WHERE rn = 1 ORDER BY region;

region | month   | amount
-------+---------+-------
N      | 2024-02 | 150   
S      | 2024-02 | 120   
(2 rows)



## 17. Procedures, functions and triggers
| | SQLite | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|
| Stored functions | none in SQL; register a function from the host language | `CREATE FUNCTION` (SQL, PL/pgSQL, others) | `CREATE FUNCTION` |
| Stored procedures | none | `CREATE PROCEDURE` + `CALL` (v11+) | `CREATE PROCEDURE` + `CALL` |
| Triggers | yes (`BEFORE/AFTER/INSTEAD OF`, row level, SQL body) | yes (row/statement level, call a function) | yes (row level) |
| Scheduled jobs | no | extension (`pg_cron`) | built-in `EVENT` |
- SQLite has only triggers inside the engine. Anything procedural lives in your application code, for example Python's `create_function`.
- PostgreSQL's triggers call a trigger function; SQLite's and MySQL's triggers have the body inline.

In [32]:
%%sql
CREATE TABLE price_log (item TEXT, old_price REAL, new_price REAL);
CREATE TABLE item (name TEXT PRIMARY KEY, price REAL);
CREATE TRIGGER trg_price AFTER UPDATE OF price ON item
BEGIN
  INSERT INTO price_log VALUES (OLD.name, OLD.price, NEW.price);
END;
INSERT INTO item VALUES ('pen', 1.5);
UPDATE item SET price = 2.0 WHERE name = 'pen';
SELECT * FROM price_log ORDER BY rowid;

item | old_price | new_price
-----+-----------+----------
pen  | 1.5       | 2.0      
(1 row)



In [33]:
# a "stored function" in SQLite lives in the host language
con.create_function("gst", 1, lambda amount: round(amount * 0.18, 2))
print(con.execute("SELECT gst(100.0)").fetchall())

[(18.0,)]


## 18. Transactions, isolation, MVCC and locking
| | SQLite | PostgreSQL 16 | MySQL 8 (InnoDB) |
|---|---|---|---|
| Concurrency control | database-level lock; one writer at a time | MVCC (readers never block writers) | MVCC + row/gap locks |
| Default isolation | serializable (effectively) | `READ COMMITTED` | `REPEATABLE READ` |
| Levels offered | via locking modes (`DEFERRED`, `IMMEDIATE`, `EXCLUSIVE`); `read_uncommitted` pragma for shared cache | `READ COMMITTED`, `REPEATABLE READ`, `SERIALIZABLE` (true SSI). `READ UNCOMMITTED` acts like read committed | all four standard levels |
| Row locks | none (the writer locks the whole database) | `SELECT ... FOR UPDATE` / `FOR SHARE`, `SKIP LOCKED`, `NOWAIT` | `FOR UPDATE` / `FOR SHARE`, `SKIP LOCKED`, `NOWAIT` (8.0+) |
| Savepoints | yes | yes | yes |
- **MVCC** = each reader sees a snapshot, so readers do not wait for writers.
- **SQLite WAL mode** lets readers continue while one writer writes. Without WAL (the default journal) a writer must briefly get exclusive access at commit time, so readers and the writer can block each other. There is still only one writer at a time.
- A plain `BEGIN` is `DEFERRED`: it takes the write lock only at the first write, so two connections can both start and then one fails with "database is locked". `BEGIN IMMEDIATE` grabs the write lock up front.
- Details of SQLite locking are in `07-Performance` and `08-SQLite-Specials`. The demo below uses two real connections to a temporary file.

In [34]:
import os, sqlite3, tempfile
path = os.path.join(tempfile.mkdtemp(), "demo.db")
a = sqlite3.connect(path, isolation_level=None, timeout=0)
b = sqlite3.connect(path, isolation_level=None, timeout=0)
a.execute("CREATE TABLE t (x INTEGER)")
a.execute("BEGIN IMMEDIATE")            # a takes the single write lock
a.execute("INSERT INTO t VALUES (1)")
try:
    b.execute("BEGIN IMMEDIATE")        # b wants it too
except sqlite3.OperationalError as e:
    print("second writer:", e)
print("reader sees:", b.execute("SELECT COUNT(*) FROM t").fetchone())   # reads still work, old snapshot
a.execute("COMMIT")
print("after commit:", b.execute("SELECT COUNT(*) FROM t").fetchone())
a.close(); b.close()

second writer: database is locked
reader sees: (0,)
after commit: (1,)


- On PostgreSQL / MySQL the second writer usually waits only if it touches the *same row* (or a conflicting lock); writers to different rows can run in parallel.
- A classic difference: a `READ COMMITTED` transaction (PostgreSQL default) can see rows other transactions committed *during* it; `REPEATABLE READ` (MySQL default) keeps one snapshot for the whole transaction.

## 19. EXPLAIN and permissions
| | SQLite | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|
| Show plan | `EXPLAIN QUERY PLAN SELECT ...` | `EXPLAIN SELECT ...` | `EXPLAIN SELECT ...` |
| Run it and time it | no (use the CLI `.timer`, `.scanstats`) | `EXPLAIN (ANALYZE, BUFFERS) ...` | `EXPLAIN ANALYZE ...` (8.0.18+) |
| Users and roles | none: whoever can open the file | roles (`CREATE ROLE`, `GRANT`, row-level security) | users (`CREATE USER`, `GRANT`), roles (8.0+) |
- `EXPLAIN ANALYZE` really **executes** the statement. In PostgreSQL, wrap `INSERT/UPDATE/DELETE` in a transaction you roll back. (MySQL's `EXPLAIN ANALYZE` accepts `SELECT` and, from 8.0.19, multi-table `UPDATE`/`DELETE`; check your version.)
- SQLite security is file security: file system permissions, encryption extensions, or doing the checks in your app.
- PostgreSQL permission model: login role + `GRANT SELECT ON t TO role`, schemas for namespaces. MySQL: `GRANT SELECT ON db.t TO 'user'@'host'`.

In [35]:
%%sql
CREATE TABLE big (id INTEGER PRIMARY KEY, k INTEGER);
CREATE INDEX ix_big_k ON big (k);
EXPLAIN QUERY PLAN SELECT id FROM big WHERE k = 5;
EXPLAIN QUERY PLAN SELECT id FROM big WHERE k + 0 = 5;

id | parent | notused | detail                                        
---+--------+---------+-----------------------------------------------
2  | 0      | 0       | SEARCH big USING COVERING INDEX ix_big_k (k=?)
(1 row)

id | parent | notused | detail  
---+--------+---------+---------
2  | 0      | 0       | SCAN big
(1 row)



- The first plan names the index (`SEARCH ... USING COVERING INDEX ix_big_k`), the second scans the table because the function on `k` hides the index. That holds on most engines: do not wrap an indexed column in a function, unless you also built an expression index. See `07-Performance`.

## 20. Concurrency model and when to pick each
- **SQLite**: a library in your process. No network, no server to run, a single file you can copy. One writer at a time, many readers. Great for apps, mobile/desktop, tests, analytics on a file, edge devices, small websites with modest write rates.
- **PostgreSQL**: a server with many concurrent writers, rich types and SQL features, extensions, strong standards compliance, strict about mistakes. A strong default for a multi-user application or complex queries.
- **MySQL**: a server, very widely deployed, often fast for simple reads, good tooling and hosting, relies on server SQL modes for strictness. Common in existing web stacks.
- Rules of thumb:
  - One process writes, data fits one machine -> SQLite.
  - Many app servers write at once, need roles, replication, extensions or heavy analytics SQL -> PostgreSQL or MySQL.
  - Prototype on SQLite only if you will test on the target engine before shipping (see `02-Writing-Portable-SQL`).
- Performance claims depend completely on workload: measure, do not assume.

## 21. Translation cheat table (32 idioms)
| # | Idiom | SQLite | PostgreSQL 16 | MySQL 8 |
|---|---|---|---|---|
| 1 | Concatenate | `a \|\| b` | `a \|\| b` | `CONCAT(a, b)` |
| 2 | Length in characters | `length(s)` | `length(s)` | `CHAR_LENGTH(s)` |
| 3 | Substring | `substr(s, 2, 3)` | `substr(s, 2, 3)` | `SUBSTRING(s, 2, 3)` |
| 4 | Position of text | `instr(s, 'x')` | `strpos(s, 'x')` | `LOCATE('x', s)` |
| 5 | Cast to text | `CAST(x AS TEXT)` | `CAST(x AS text)` / `x::text` | `CAST(x AS CHAR)` |
| 6 | Null default | `coalesce(x, 0)` | `coalesce(x, 0)` | `COALESCE(x, 0)` (`IFNULL`) |
| 7 | Null-safe equal | `a IS b` | `a IS NOT DISTINCT FROM b` | `a <=> b` |
| 8 | Case-insensitive match | `lower(a) = lower(b)` | `a ILIKE b` | default collation already ci |
| 9 | Regex match | needs a registered `REGEXP` function | `a ~ 'pat'` | `a REGEXP 'pat'` |
| 10 | Integer division | `a / b` (two integers) | `a / b` (two integers) | `a DIV b` |
| 11 | Random number | `random()` (big integer) | `random()` (0 to 1) | `RAND()` (0 to 1) |
| 12 | Current timestamp | `CURRENT_TIMESTAMP` | `now()` | `NOW()` |
| 13 | Format a date | `strftime('%Y-%m', d)` | `to_char(d, 'YYYY-MM')` | `DATE_FORMAT(d, '%Y-%m')` |
| 14 | Add days | `date(d, '+7 day')` | `d + interval '7 days'` | `DATE_ADD(d, INTERVAL 7 DAY)` |
| 15 | Days between | `julianday(a) - julianday(b)` | `a - b` (dates) | `DATEDIFF(a, b)` |
| 16 | First of month | `date(d, 'start of month')` | `date_trunc('month', d)` | `DATE_FORMAT(d, '%Y-%m-01')` |
| 17 | First n rows | `LIMIT n` | `LIMIT n` / `FETCH FIRST n ROWS ONLY` | `LIMIT n` |
| 18 | Page | `LIMIT n OFFSET m` | `LIMIT n OFFSET m` | `LIMIT n OFFSET m` |
| 19 | String aggregate | `group_concat(x, ',')` | `string_agg(x, ',')` | `GROUP_CONCAT(x SEPARATOR ',')` |
| 20 | Conditional count | `count(*) FILTER (WHERE c)` | `count(*) FILTER (WHERE c)` | `SUM(CASE WHEN c THEN 1 ELSE 0 END)` |
| 21 | Upsert | `ON CONFLICT (k) DO UPDATE SET v = excluded.v` | same with `EXCLUDED.v` | `ON DUPLICATE KEY UPDATE v = VALUES(v)` (8.0.20+: row alias `AS new ... v = new.v`) |
| 22 | Insert, skip duplicates | `INSERT OR IGNORE` / `ON CONFLICT DO NOTHING` | `ON CONFLICT DO NOTHING` | `INSERT IGNORE` |
| 23 | Get new id | `RETURNING id` / `last_insert_rowid()` | `RETURNING id` | `LAST_INSERT_ID()` |
| 24 | Auto key | `INTEGER PRIMARY KEY` | `GENERATED ... AS IDENTITY` | `AUTO_INCREMENT` |
| 25 | Quote identifier | `"name"` | `"name"` | `` `name` `` |
| 26 | Update from another table | `UPDATE t SET c = u.c FROM u WHERE ...` | same | `UPDATE t JOIN u ON ... SET t.c = u.c` |
| 27 | Empty a table | `DELETE FROM t` | `TRUNCATE t` | `TRUNCATE TABLE t` |
| 28 | Latest row per group | `ROW_NUMBER() OVER (...)` | `DISTINCT ON (g)` or `ROW_NUMBER()` | `ROW_NUMBER() OVER (...)` |
| 29 | List tables | `SELECT name FROM sqlite_master WHERE type='table'` | `information_schema.tables` / `\dt` | `SHOW TABLES` |
| 30 | Describe a table | `PRAGMA table_info(t)` | `information_schema.columns` / `\d t` | `DESCRIBE t` |
| 31 | Show the plan | `EXPLAIN QUERY PLAN` | `EXPLAIN` (`EXPLAIN ANALYZE` also runs it) | `EXPLAIN` (`EXPLAIN ANALYZE` also runs it, 8.0.18+) |
| 32 | Full outer join | `FULL JOIN` (3.39+) | `FULL JOIN` | `LEFT JOIN ... UNION ... RIGHT JOIN` |
- Items marked `/` give two valid options. Verify exact forms for your version.

In [36]:
%%sql
-- a few rows of the table, proven on SQLite
SELECT 'a' || 'b' AS r1, length('héllo') AS r2, substr('hello', 2, 3) AS r3, instr('hello', 'l') AS r4,
       CAST(42 AS TEXT) AS r5, coalesce(NULL, 0) AS r6, (NULL IS NULL) AS r7_is_null_equal,
       date('2024-03-15', '+7 day') AS r14, date('2024-03-15', 'start of month') AS r16;

r1 | r2 | r3  | r4 | r5 | r6 | r7_is_null_equal | r14        | r16       
---+----+-----+----+----+----+------------------+------------+-----------
ab | 5  | ell | 3  | 42 | 0  | 1                | 2024-03-22 | 2024-03-01
(1 row)



## 22. Summary
- Learn the core once, then keep this table next to you for strings, dates, upsert, paging, JSON and DDL.
- Compile-only checks (SQLAlchemy) prove that *text* is generated, not that the target engine accepts it.
- Next: `02-Writing-Portable-SQL` for how to avoid most of these differences, and how to move from SQLite to PostgreSQL safely.